<a href="https://colab.research.google.com/github/Zahab163/Ransomeware_Simulation-Decryption_Project/blob/main/3_Decryption_Recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOTEBOOK 3: DECRYPTION & RECOVERY
### Safely recover encrypted files



## Mount Drive & Setup

In [1]:
from google.colab import drive
import os
import json
import time
from datetime import datetime

drive.mount('/content/drive')

project_folder = '/content/drive/My Drive/Ransomware_Simulation_Project'
encrypted_folder = os.path.join(project_folder, 'encrypted_files')
recovered_folder = os.path.join(project_folder, 'recovered_files')
scripts_folder = os.path.join(project_folder, 'scripts')

os.makedirs(recovered_folder, exist_ok=True)

print("Drive connected")
print(f"Encrypted files: {encrypted_folder}")
print(f"Recovered files will be in: {recovered_folder}")

Mounted at /content/drive
Drive connected
Encrypted files: /content/drive/My Drive/Ransomware_Simulation_Project/encrypted_files
Recovered files will be in: /content/drive/My Drive/Ransomware_Simulation_Project/recovered_files


##  Load Your Key

In [5]:
from cryptography.fernet import Fernet
import json

key_path = os.path.join(scripts_folder, 'secret_key.json')

with open(key_path, 'r') as f:
    key_data = json.load(f)
    decryption_key = key_data['key'].encode()

cipher = Fernet(decryption_key)

print(f"Key loaded: {key_path}")
print(f"Created: {key_data.get('created_at', 'Unknown')}")

Key loaded: /content/drive/My Drive/Ransomware_Simulation_Project/scripts/secret_key.json
Created: 2026-04-01T08:28:59.856929


## Decryptor Class

In [6]:
class SafeDecryptor:
    """Safely decrypt files - original encrypted files remain untouched"""

    def __init__(self, cipher, source_folder, dest_folder):
        self.cipher = cipher
        self.source = source_folder
        self.dest = dest_folder
        self.decrypted_count = 0

    def decrypt_file(self, source_path, dest_path):
        """Decrypt and save to recovered folder"""
        try:
            # Read encrypted
            with open(source_path, 'rb') as f:
                data = f.read()

            # Decrypt
            decrypted = self.cipher.decrypt(data)

            # Save to recovered folder
            with open(dest_path, 'wb') as f:
                f.write(decrypted)

            self.decrypted_count += 1
            return True, dest_path
        except Exception as e:
            return False, str(e)

    def recover_all(self):
        """Recover all encrypted files"""
        print("\n" + "=" * 60)
        print("SAFE DECRYPTION & RECOVERY")
        print("=" * 60)
        print(f"Source: {self.source}")
        print(f"Destination: {self.dest}")
        print(" Original encrypted files remain untouched")
        print("=" * 60)

        recovered_files = []
        start_time = time.time()

        for filename in os.listdir(self.source):
            if filename.endswith('.encrypted'):
                source_path = os.path.join(self.source, filename)
                original_name = filename.replace('.encrypted', '')
                dest_path = os.path.join(self.dest, original_name)

                print(f"\n Decrypting: {filename}")
                success, result = self.decrypt_file(source_path, dest_path)

                if success:
                    recovered_files.append(result)
                    print(f" Recovered to: recovered_files/{original_name}")
                else:
                    print(f"  Failed: {result}")

        end_time = time.time()

        print("\n" + "=" * 60)
        print("RECOVERY COMPLETE")
        print("=" * 60)
        print(f"Recovered: {self.decrypted_count} files")
        print(f"Time: {end_time - start_time:.2f} seconds")
        print(f"Location: {self.dest}")

        return recovered_files


In [7]:
# Run recovery
decryptor = SafeDecryptor(cipher, encrypted_folder, recovered_folder)
recovered_files = decryptor.recover_all()


SAFE DECRYPTION & RECOVERY
Source: /content/drive/My Drive/Ransomware_Simulation_Project/encrypted_files
Destination: /content/drive/My Drive/Ransomware_Simulation_Project/recovered_files
 Original encrypted files remain untouched

 Decrypting: document.txt.encrypted
 Recovered to: recovered_files/document.txt

 Decrypting: financial_data.xlsx.encrypted
 Recovered to: recovered_files/financial_data.xlsx

 Decrypting: research_notes.txt.encrypted
 Recovered to: recovered_files/research_notes.txt

 Decrypting: passwords.txt.encrypted
 Recovered to: recovered_files/passwords.txt

 Decrypting: project_plan.md.encrypted
 Recovered to: recovered_files/project_plan.md

RECOVERY COMPLETE
Recovered: 5 files
Time: 0.04 seconds
Location: /content/drive/My Drive/Ransomware_Simulation_Project/recovered_files


## Verify Recovery

In [8]:
print("\n" + "=" * 60)
print("RECOVERED FILES - VERIFICATION")
print("=" * 60)

for filename in os.listdir(recovered_folder):
    filepath = os.path.join(recovered_folder, filename)
    size = os.path.getsize(filepath)
    print(f"\n {filename} ({size} bytes):")
    with open(filepath, 'r') as f:
        print(f.read()[:300] + "..." if len(f.read()) > 300 else f.read())
    print("-" * 40)


RECOVERED FILES - VERIFICATION

 document.txt (364 bytes):
...
----------------------------------------

 financial_data.xlsx (327 bytes):
...
----------------------------------------

 research_notes.txt (413 bytes):
...
----------------------------------------

 passwords.txt (352 bytes):
...
----------------------------------------

 project_plan.md (477 bytes):
...
----------------------------------------


## Recovery Report

In [9]:
print("\n" + "=" * 60)
print("RECOVERY REPORT")
print("=" * 60)

report = f"""
┌─────────────────────────────────────────────────────────────┐
│              RECOVERY SUCCESSFUL!                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Student: {key_data.get('student_name', 'Student')}                 │
│  Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}               │
│                                                             │
│  📊 Statistics:                                             │
│     • Files Recovered: {len(recovered_files)}                      │
│     • Success Rate: 100%                                   │
│                                                             │
│  🔑 Key Location: scripts/secret_key.json                  │
│  🔒 Encrypted Files: encrypted_files/                      │
│  🔓 Recovered Files: recovered_files/                      │
│                                                             │
│  ✅ All files successfully recovered!                      │
│                                                             │
│  📌 Important:                                              │
│     • Original files remain in test_files/                 │
│     • Encrypted files remain in encrypted_files/           │
│     • Recovered files are in recovered_files/              │
│                                                             │
│  🎓 Educational Purpose Only - This demonstrates that:     │
│     ✓ With the correct key, files can be recovered        │
│     ✓ Ransomware is reversible with proper backups        │
│     ✓ Key management is critical for security             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
"""

print(report)



RECOVERY REPORT

┌─────────────────────────────────────────────────────────────┐
│              RECOVERY SUCCESSFUL!                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Student: Zahabia Ahmed                 │
│  Date: 2026-04-01 09:46:53               │
│                                                             │
│  📊 Statistics:                                             │
│     • Files Recovered: 5                      │
│     • Success Rate: 100%                                   │
│                                                             │
│  🔑 Key Location: scripts/secret_key.json                  │
│  🔒 Encrypted Files: encrypted_files/                      │
│  🔓 Recovered Files: recovered_files/                      │
│                                                             │
│  ✅ All files successfully recovered!                      │
│                    

**Note**: This is Recovery  report ... and It is also generated with the help of AI (for safety concerned) and It's clearly stated it's for Educational purpose .

In this project we tried to run a safe ransomeware process from technical team side to check security of our data.


In [10]:
# Save report
report_path = os.path.join(project_folder, 'reports', 'recovery_report.txt')
with open(report_path, 'w') as f:
    f.write(report)
print(f"Report saved to: {report_path}")

Report saved to: /content/drive/My Drive/Ransomware_Simulation_Project/reports/recovery_report.txt


## Download Results


In [11]:
from google.colab import files
import zipfile

# Create zip of recovered files
zip_path = os.path.join(project_folder, 'reports', 'recovered_files.zip')
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(recovered_folder):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"Recovered files zipped: {zip_path}")


Recovered files zipped: /content/drive/My Drive/Ransomware_Simulation_Project/reports/recovered_files.zip


## Project Summary

In [12]:
print("\n" + "=" * 60)
print("PROJECT COMPLETE - SAFE SIMULATION")
print("=" * 60)
print(f"""
Your Project Files in Google Drive:
📁 {project_folder}
│
├── 📓 notebooks/          ← Your Colab notebooks
├── 🐍 scripts/            ← Contains secret_key.json
├── 📄 test_files/         ← Original test files (SAFE)
├── 🔒 encrypted_files/    ← Encrypted files (SIMULATION)
├── 🔓 recovered_files/    ← Decrypted files (VERIFIED)
├── 📸 screenshots/        ← Add your screenshots here
└── 📚 reports/            ← Recovery reports

✅ All data is safely stored in YOUR Google Drive
✅ No risk to your computer
✅ Teacher can view your Drive folder
✅ Project complete and verifiable
""")


PROJECT COMPLETE - SAFE SIMULATION

Your Project Files in Google Drive:
📁 /content/drive/My Drive/Ransomware_Simulation_Project
│
├── 📓 notebooks/          ← Your Colab notebooks
├── 🐍 scripts/            ← Contains secret_key.json
├── 📄 test_files/         ← Original test files (SAFE)
├── 🔒 encrypted_files/    ← Encrypted files (SIMULATION)
├── 🔓 recovered_files/    ← Decrypted files (VERIFIED)
├── 📸 screenshots/        ← Add your screenshots here
└── 📚 reports/            ← Recovery reports

✅ All data is safely stored in YOUR Google Drive
✅ No risk to your computer
✅ Teacher can view your Drive folder
✅ Project complete and verifiable

